# 🌸 Sakuga4K — 4K Anime Video Upscaler Cloud Engine
### Free Ephemeral Nvidia T4 GPU Backend for AMV & Anime Video Editors

---

### ⚡ Step 1: Click "Run all" (Runtime ➔ Run all)
This single step will:
1. Verify your free Nvidia T4 GPU (16 GB VRAM).
2. Mount your Google Drive for permanent 4K auto-sync.
3. Start the FastAPI neural backend in half-precision (FP16).
4. Generate your **Cloudflare Quick Tunnel URL** and **Scannable QR Code**.

Copy the printed link and paste it into your Sakuga4K web studio!

In [ ]:
# @title 🚀 1-Click Master Engine (Hit Play Button)
!nvidia-smi

# 1. Mount Google Drive for automatic permanent output sync
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# 2. Clone / Pull Repository & Install Dependencies
import os, sys
if not os.path.exists('/content/Sakuga4k'):
    !git clone https://github.com/Anyone-design/Sakuga4k.git /content/Sakuga4k
os.chdir('/content/Sakuga4k')
!git pull
!pip install -q -r backend/requirements.txt qrcode[pil]

# 3. Install Cloudflare Quick Tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!rm -f cloudflared-linux-amd64.deb

# 4. Launch FastAPI Uvicorn in Background
import subprocess, time, re, qrcode
server_process = subprocess.Popen(
    ["uvicorn", "backend.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(3)

# 5. Establish Tunnel & Print QR / Link
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True
)

print("\n=======================================================")
print("🌸 Sakuga4K Cloud Tunnel Starting...")
print("=======================================================\n")

tunnel_url = None
for line in tunnel_process.stdout:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    print("\n✅ SUCCESS! Paste this URL into your Sakuga4K Studio Web App:")
    print(f"\n👉  {tunnel_url}  👈\n")
    
    # Print ASCII QR code for instant mobile phone pairing
    qr = qrcode.QRCode()
    qr.add_data(tunnel_url)
    qr.print_ascii(invert=True)
    print("\nKeep this Colab tab running while upscaling clips!")
else:
    print("❌ Tunnel failed to start. Check /content/uvicorn.log")
